# Tapi · Exploración inicial con SQL

Ejecutá las celdas en orden. Usamos DuckDB para consultar los CSV con SQL dentro de Colab. Las tablas se llaman cuentas, consultas y pagos. Esta primera exploración cubre muestras, volumen, identificadores y convenios.

En Colab, cargá los tres CSV cuando aparezca el selector. Si ejecutás el notebook desde la carpeta del repositorio, se leen desde data/.

In [ ]:
%pip -q install duckdb==1.4.3

In [ ]:
from pathlib import Path
import duckdb

nombres = {
    "cuentas": "B_agenda_cuentas.csv",
    "consultas": "B_agenda_consultas.csv",
    "pagos": "B_agenda_pagos.csv",
}
carpeta = Path("data") if Path("data").is_dir() else Path(".")
faltantes = [nombre for nombre in nombres.values() if not (carpeta / nombre).exists()]
if faltantes:
    from google.colab import files
    print("Seleccioná los tres archivos CSV del challenge.")
    files.upload()
    carpeta = Path(".")
    faltantes = [nombre for nombre in nombres.values() if not (carpeta / nombre).exists()]
    if faltantes:
        raise FileNotFoundError(f"Faltan archivos: {faltantes}")

conexion = duckdb.connect()
for tabla, archivo in nombres.items():
    ruta = str(carpeta / archivo).replace("'", "''")
    conexion.execute(
        f"CREATE OR REPLACE TABLE {tabla} AS "
        f"SELECT * FROM read_csv('{ruta}', header=true, all_varchar=true)"
    )

def consultar(sql):
    return conexion.execute(sql).df()


Cargamos inicialmente todas las columnas como texto para conservar IDs y timestamps originales. Los campos vacíos se leen como NULL. Más adelante convertiremos fechas y montos para los cálculos que lo requieran.

## Muestra de cuentas

Cada fila representa una adhesión a la Agenda. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM cuentas
LIMIT 10
""")

## Muestra de consultas

Cada fila representa un intento de consulta. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM consultas
LIMIT 10
""")

## Muestra de pagos

Cada fila representa un pago realizado. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM pagos
LIMIT 10
""")

## Cantidad de registros

COUNT(*) cuenta filas. UNION ALL apila los resultados de las tres tablas.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, COUNT(*) AS filas FROM cuentas
UNION ALL
SELECT 'consultas' AS tabla, COUNT(*) AS filas FROM consultas
UNION ALL
SELECT 'pagos' AS tabla, COUNT(*) AS filas FROM pagos
""")

## Identificadores de cuentas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(account_id) AS filas_con_id,
    COUNT(DISTINCT account_id) AS cuentas_distintas
FROM cuentas
""")

## Identificadores de consultas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(consulta_id) AS filas_con_id,
    COUNT(DISTINCT consulta_id) AS consultas_distintas
FROM consultas
""")

## Identificadores de pagos

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(pago_id) AS filas_con_id,
    COUNT(DISTINCT pago_id) AS pagos_distintos
FROM pagos
""")

## Convenios y proveedores

Una fila por combinación de convenio y proveedor. Permite identificar convenios compartidos y comparar el volumen de consultas.

In [ ]:
consultar("""
SELECT
    convenio,
    proveedor,
    COUNT(*) AS cantidad_consultas
FROM consultas
GROUP BY convenio, proveedor
ORDER BY convenio, proveedor
""")

## Tus observaciones

Anotá qué encontraste en cada consulta y qué querés investigar después. Revisaremos origen, resultado, fechas y campos vacíos en el próximo paso.